# 1. Problem Statement & Goals 🎯
___
## Problem Statement
Ebuss, a growing e-commerce company with a significant market share in categories like household essentials, personal care, and electronics, aims to scale rapidly and compete with market leaders like Amazon and Flipkart.

To achieve this, Ebuss needs to leverage its vast data on user reviews and ratings. As a Senior Machine Learning Engineer, the core challenge is to build a sentiment-based product recommendation system. This system must not only recommend products based on user behaviors (ratings) but also refine those recommendations by analyzing the sentiment of the textual reviews associated with those products. The ultimate objective is to enhance the user experience by suggesting products that users are most likely to purchase and feel positive about.

## Goals
The project is divided into four main objectives to achieve the problem statement:

### 1. Data Sourcing and Sentiment Analysis

#### Objective: 
* Build a Machine Learning model to classify user reviews as Positive or Negative.

#### Key Tasks:

* Perform Exploratory Data Analysis (EDA), data cleaning, and text preprocessing.

* Extract features using techniques like Bag-of-Words, TF-IDF, or Word Embeddings.

* Train and evaluate at least three of the following classification models: Logistic Regression, Random Forest, XGBoost, or Naive Bayes.

* Select the best-performing model to predict user sentiment.

### 2. Building a Recommendation System

#### Objective: 
* Identify the most effective recommendation technique for the dataset.

#### Key Tasks:

* Develop both User-based and Item-based collaborative filtering recommendation systems.

* Analyze and compare their performance to select the best-suited system.

* Generate an initial list of 20 recommended products for a specific user based on their historical ratings.

### 3. Improving Recommendations using Sentiment Analysis

#### Objective: 
* Create a hybrid "Sentiment-Based Recommendation System."

#### Key Tasks:

* Integrate the chosen Sentiment Analysis model with the Recommendation System.

* Take the top 20 products recommended by the collaborative filtering system.

* Filter and rank these products based on their predicted sentiment scores.

* Output the final top 5 products that have the highest positive sentiment.

### 4. Deployment

#### Objective: 
* Make the solution accessible via a web interface.

#### Key Tasks:

* Build a web application using the Flask framework.

* Create a User Interface (UI) that accepts a username and displays the top 5 recommended products.

* Deploy the end-to-end application (Model + API + UI) on a cloud platform like Heroku.

In [ ]:
import pandas as pd

import notebook_setup
from notebook_setup import ROOT_DIR, DATA_DIR, ROW_DATA_DIR, PROCESSED_DATA_DIR

from src.utils.data_loader import load_data

from src.utils.generic_methods import get_column_value_counts, get_missing_columns

from src.config.manager import ConfigManager

config_manager = ConfigManager()

# 2. Setup and Data Loading ⚙️

In [ ]:
# Assign the returned pandas DataFrame from the 'load_data' function to the variable 'df'.
# The function is passed a file path, which is constructed in a platform-independent way
# using the pathlib library. The '/' operator joins the 'ROW_DATA_DIR' Path object
# with the filename "dataset.csv".
df_raw = load_data(ROW_DATA_DIR / "dataset.csv")

## Initial Inspection

### 📘 Dataset Attribute Description

| Attribute              | Description                                                                                                                           |
|------------------------|---------------------------------------------------------------------------------------------------------------------------------------|
| **id**                 | Unique identity number to identify each review given by a user to a particular product in the dataset.                                |
| **brand**              | Name of the brand of the product being reviewed.                                                                                      |
| **categories**         | Category of the product (e.g., household essentials, books, personal care, medicines, cosmetics, beauty products, appliances, etc.). |
| **manufacturer**       | Name of the manufacturer of the product.                                                                                              |
| **name**               | Name of the product for which the review or rating was added.                                                                          |
| **reviews_date**       | Date on which the review was added by the user.                                                                                       |
| **reviews_didPurchase**| Indicates whether the user purchased the product.                                                                                     |
| **reviews_doRecommend**| Indicates whether the user recommends the product.                                                                                    |
| **reviews_rating**     | Rating given by the user to the product.                                                                                              |
| **reviews_text**       | Text of the review written by the user.                                                                                               |
| **reviews_title**      | Title of the review provided by the user.                                                                                             |
| **reviews_userCity**   | City where the user resides.                                                                                                          |
| **reviews_userProvince**| Province/state where the user resides.                                                                                               |
| **reviews_username**   | Unique identifier for the individual user in the dataset.                                                                             |
| **user_sentiment**     | Overall sentiment of the user for the product (Positive or Negative).                                                                  |


In [ ]:
# This will print the first 5 rows of the DataFrame to the console
display(df_raw.head())

In [ ]:
# This will print the last 5 rows of the DataFrame to the console
display(df_raw.tail())

In [ ]:
# Print dimensionality
rows, cols = df_raw.shape
print(f"✅ Dataframe Shape: {rows:,} rows x {cols} columns")

In [ ]:
# Print schema and non-null counts
print("\n📊 Dataframe Info:")
# verbose=True ensures full info is printed even for large DataFrames
df_raw.info(verbose=True, show_counts=True)

In [ ]:
missing_value_columns = get_missing_columns(df_raw)
config_manager.set_config("missing_value_columns", missing_value_columns)

In [ ]:
invalid_columns = ['id', ]
config_manager.set_config("invalid_columns", invalid_columns)

### **Attribute Importance for Recommendation System**

To effectively build the sentiment-based recommendation system for Ebuss, it's crucial to understand the role of each attribute in our dataset.

#### **Highly Important Attributes:**

*   **`brand`**: Essential for understanding product characteristics and grouping products. Will be vital for content-based recommendations and insights into brand perception.
*   **`categories`**: Critical for identifying product types, filtering recommendations, and understanding consumer preferences across different product segments.
*   **`manufacturer`**: Similar to `brand`, this provides valuable product context and can be used for grouping and recommending similar products.
*   **`name`**: The product's name is crucial for identification and presentation in recommendations.
*   **`reviews_didPurchase`**: This boolean attribute is extremely valuable. Knowing if a user actually purchased the product provides a strong signal of intent and satisfaction, which can be leveraged to validate recommendation models.
*   **`reviews_doRecommend`**: Directly addresses a key project goal: predicting whether a user will recommend a product. This will serve as a target variable or a strong feature in our models.
*   **`reviews_rating`**: A primary explicit feedback mechanism. Ratings are fundamental for sentiment analysis, collaborative filtering components, and assessing user satisfaction.
*   **`reviews_text`**: The textual content of reviews is paramount for sentiment analysis, which is a core component of this project. It provides rich, granular insights into user opinions.
*   **`reviews_title`**: Often encapsulates the main sentiment or summary of a review, making it a valuable input for sentiment analysis alongside `reviews_text`.
*   **`user_sentiment`**: Explicitly stated in the problem statement as an attribute to leverage. This pre-calculated sentiment will be a direct input for the sentiment-based recommendation model and a key output for analysis.

#### **Less Important Attributes (why?):**

*   **`id`**: While a unique identifier, it does not directly contribute to the *logic* of the recommendation system itself. It's useful for data management but was already dropped as part of the initial cleanup, which is appropriate for modeling purposes.
*   **`reviews_date`**: The date of the review can be useful for time-series analysis or understanding trends, but for the core sentiment-based recommendation logic outlined in the goals, its direct utility as a *feature* is secondary. It wasn't dropped but its role in the core model might be limited initially.
*   **`reviews_userCity`**, **`reviews_userProvince`**, **`reviews_username`**: These attributes were dropped in a following step. This is suitable as the current project goals focus on product-centric recommendations enhanced by sentiment, rather than highly localized or individual user-profile-based recommendations that would require these specific identifiers. While `reviews_username` could be used for user-based collaborative filtering, the emphasis here is on sentiment and product characteristics, making these less critical for the initial phase.

#### **Conclusion:**

The project will heavily rely on `brand`, `categories`, `manufacturer`, `reviews_didPurchase`, `reviews_doRecommend`, `reviews_rating`, `reviews_text`, `reviews_title`, and `user_sentiment` to achieve its goals of building a robust sentiment-based recommendation system. The `name` attribute will be used for displaying recommended products. The dropped user-specific and location-specific attributes align with the current scope, which prioritizes leveraging product feedback and sentiment.

# 3. Data Cleaning and Preprocessing 🛠️

In [ ]:
existing_cols = set(df_raw.columns)
target_cols = set(['id', 'reviews_userCity', 'reviews_userProvince'])
cols_to_drop_verified = list(target_cols.intersection(existing_cols))
missing_cols = target_cols - existing_cols

if missing_cols:
    print(f"The following columns were not found and will be ignored: {missing_cols}")

if not cols_to_drop_verified:
    print("No matching columns found to drop. Returning original DataFrame.")
    df_cleaned = df_raw.copy()

else:
    try:
        df_cleaned = df_raw.drop(columns=cols_to_drop_verified)
        print(f"Dropped columns: {cols_to_drop_verified}")
        print(f"Final DataFrame shape: {df_cleaned.shape}")
        display(df_cleaned.head())
    except Exception as e:
        print(f"An unexpected error occurred during column dropping: {e}")
        raise

# 4. Univariate Analysis (Distribution Analysis) 📈

# 5. Bivariate & Multivariate Analysis (Feature Relationships) 🤝

# 6. Text-Specific EDA (Natural Language Processing) 💬

# 7. Summary and Key Insights ✨